### Tools

##### Models can request to call tools that perform tasks such as fetching datas from a database, searching the web, or running code.
Tools are pairings of : 

1. A schema, including the name of the tool, a description, and or arguments definitions(often a JSON schema)
2. A function or coroutine to execute.

In [22]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

model = ChatOpenAI(model="gpt-5.4-mini")

In [23]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather in a given location."""
    return f"The weather in {location} is sunny."


model_with_tool = model.bind_tools([get_weather])

In [24]:
response = model_with_tool.invoke("What is the weather in New York?")
print(response)

print(f"Response content: {response.content}")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 135, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-E6ja2As08yojM3h47ziJkYEKMvQxB', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019faa9d-a3f9-7d40-971e-1a6597362f2b-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_1XN2DgTxkeSHQ52IbS3fUL5J', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 135, 'output_tokens': 18, 'total_tokens': 153, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Res

##### Tool Execution Loops

In [25]:
# step 1 : model generates the tool call instead of answering directly
from langchain.messages import HumanMessage

messages = [HumanMessage("What is the weather in New York?")]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)


final_response = model_with_tool.invoke(messages)
print(f"Final response content: {final_response.content}")

Final response content: The weather in New York is sunny.


In [26]:
messages

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 135, 'total_tokens': 153, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-E6ja4AGC8LdoURrXFSS6RDGwtC2Sk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019faa9d-aaa9-79b2-b9b4-53f9008c3dd5-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_JpxunOd5y8w4x5D7GdTbslLs', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 18, 'total_token